# LoRA для prompt-guard: дообучение xlm-roberta-base

Обучает небольшие надстройки (LoRA) поверх `xlm-roberta-base` для классификации «атака / безопасно».
Базовая модель остаётся замороженной, обучаются только матрицы LoRA и классификационная голова.

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Запускайте ячейки сверху вниз.
3. В ячейке загрузки выберите **два** файла из `data/splits/`: `train.jsonl` и `val.jsonl`. **golden.jsonl сюда не загружайте**: на нём модель не должна обучаться.
4. В конце скачается `lora_adapter.zip` (несколько мегабайт). Распакуйте его в `results/lora-xlmr/adapter/` проекта.

Остановка обучения и выбор лучшей эпохи идут по val.

In [ ]:
!pip install -q peft transformers accelerate
!pip uninstall -y -q torchao  # в Colab старая версия torchao ломает peft, а нам он не нужен
import torch
assert torch.cuda.is_available(), "включите GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()  # выберите train.jsonl и val.jsonl
for name in ("train.jsonl", "val.jsonl"):
    assert name in uploaded, f"не загружен {name}"
assert "golden.jsonl" not in uploaded, "golden нельзя использовать при обучении"
print({k: len(v) for k, v in uploaded.items()})

In [ ]:
import json
import numpy as np
from datasets import Dataset

MODEL = "xlm-roberta-base"
MAX_LEN = 384  # самый длинный текст в train: 332 токена, обрезки не будет
SEED = 13

def load(name):
    rows = [json.loads(l) for l in uploaded[name].decode("utf-8").splitlines() if l.strip()]
    return Dataset.from_dict({"text": [r["text"] for r in rows], "label": [int(r["label"] == "attack") for r in rows]})

train_ds, val_ds = load("train.jsonl"), load("val.jsonl")
print(train_ds, val_ds, "доля атак в train:", np.mean(train_ds["label"]).round(3))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

tok = AutoTokenizer.from_pretrained(MODEL)
tokenize = lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN)
train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["text"])

base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1,
                  target_modules=["query", "value"])  # голова классификатора обучается тоже
model = get_peft_model(base, lora)
model.print_trainable_parameters()

In [ ]:
import inspect
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, DataCollatorWithPadding
from sklearn.metrics import f1_score, recall_score, precision_score

def metrics(p):
    pred = p.predictions.argmax(-1); y = p.label_ids
    neg = max(1, int((y == 0).sum()))
    return {"f1": f1_score(y, pred), "recall": recall_score(y, pred), "precision": precision_score(y, pred),
            "fpr": float(((pred == 1) & (y == 0)).sum() / neg)}

# в новых версиях transformers warmup_ratio и group_by_length заменены, поэтому аргументы подбираются по версии
accepted = inspect.signature(TrainingArguments.__init__).parameters
kw = dict(output_dir="out", seed=SEED, num_train_epochs=8, per_device_train_batch_size=16, per_device_eval_batch_size=64,
          learning_rate=3e-4, weight_decay=0.01, fp16=True, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
          load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True, logging_steps=50, report_to="none")
kw.update({"warmup_steps": 0.06} if "warmup_ratio" not in accepted else {"warmup_ratio": 0.06})
kw.update({"train_sampling_strategy": "group_by_length"} if "train_sampling_strategy" in accepted else {"group_by_length": True})
args = TrainingArguments(**kw)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()
print("лучшая эпоха по val F1:", trainer.state.best_metric)

In [ ]:
# Оценка после обучения. golden загружается только сейчас, когда обучение и выбор лучшей эпохи уже закончены.
import hashlib, os, torch
uploaded_g = files.upload()  # выберите golden.jsonl
assert "golden.jsonl" in uploaded_g

@torch.no_grad()
def attack_probs(texts, batch=64):
    model.eval(); out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(model.device)
        out += torch.softmax(model(**enc).logits.float(), dim=-1)[:, 1].cpu().tolist()
    return out

os.makedirs("lora_adapter", exist_ok=True)
for name, raw in (("val", uploaded["val.jsonl"]), ("golden", uploaded_g["golden.jsonl"])):
    texts = [json.loads(l)["text"] for l in raw.decode("utf-8").splitlines() if l.strip()]
    probs = attack_probs(texts)
    json.dump({"split": name, "split_sha256": hashlib.sha256(raw).hexdigest(), "n": len(texts), "probs": [round(p, 6) for p in probs]},
              open(f"lora_adapter/probs_{name}.json", "w"))
    print(name, len(texts), "готово")

In [ ]:
import shutil
history = [h for h in trainer.state.log_history if "eval_f1" in h]
json.dump({"model": MODEL, "max_len": MAX_LEN, "lora": {"r": 16, "alpha": 32, "dropout": 0.1, "target": ["query", "value"]},
           "lr": 3e-4, "seed": SEED, "best_val_f1": trainer.state.best_metric, "val_history": history},
          open("train_log.json", "w"), ensure_ascii=False, indent=2)
model.save_pretrained("lora_adapter")  # только надстройки LoRA и голова, базовая модель не сохраняется
tok.save_pretrained("lora_adapter")
shutil.copy("train_log.json", "lora_adapter/train_log.json")
shutil.make_archive("lora_adapter", "zip", "lora_adapter")
files.download("lora_adapter.zip")